# 🎤 N-Gram Karaoke

Welcome to **N-Gram Karaoke**!

In this workshop, you’ll explore n‑gram language models and use them to generate new poems and songs from your chosen corpus.

## How it works

An n-gram model learns patterns from a text by looking at sequences of words. 
Given the last *n* words it has seen, it predicts what word might come next — 
based entirely on what it observed in the original text.

For example, with n=2, if the model sees the word **"never"**, it might follow it 
with **"gonna"** because it saw "never gonna" many times in the corpus.
The larger the n, the more the output resembles the original text. 
The smaller the n, the more creative (and chaotic) it gets!

## What you'll do

- 🎵 Use Genius API to save song lyrics and make your own **corpus** (the text the model learns from)
- 🎨 **Mix artists** together to create unexpected mashups
- 🎛️ Experiment with different **n-gram sizes** to see how it changes the output
- 🏆 Generate your best lyrics and share them with the group!

## Tips before you start

- If the output is **too repetitive** → lower your n value
- If the output is **too random** → raise your n value  
- If the output is **too similar to the original** → lower your n value
- Sweet spot is usually **n=2 or n=3** for small corpora

Let's generate some bangers! 🎶

# Making your corpus

The following cell uses the LyricsGenius library in Python, which allows us to easily get lyrics through Genius' Application Programming Interface (API)!

Run the code below and figure out what each function does, then replace some of the arguments to find artists and songs you like. Here's the library's documentation for reference: https://lyricsgenius.readthedocs.io/en/master/usage.html

In [71]:
# RUN THIS LINE ONCE THEN COMMENT IT AWAY!
#!pip install lyricsgenius 
from lyricsgenius import Genius
import os

token = open("token.txt").read()

genius = Genius(token)

genius.remove_section_headers = True # what happens when this is False?

artist = genius.search_artist("Led Zeppelin", max_songs=3, sort="popularity")
song = genius.search_song("Good Times Bad Times", artist.name)
album = genius.search_album("Led Zeppelin I", "Led Zeppelin")

print(artist.songs)
print(song.lyrics)

[Song(title=Stairway to Heaven, artist=Led Zeppelin), Song(title=Immigrant Song, artist=Led Zeppelin), Song(title=Going to California, artist=Led Zeppelin)]
In the days of my youth, I was told what it means to be a man
Now I've reached that age, I've tried to do all those things the best I can
No matter how I try, I find my way into the same old jam

Good times, bad times, you know I've had my share
When my woman left home for a brown-eyed man
But I still don't seem to care

Sixteen, I fell in love with a girl as sweet as could be
It only took a couple of days 'til she was rid of me
She swore that she would be all mine and love me 'til the end
But when I whispered in her ear, I lost another friend, oh

Good times, bad times, you know I've had my share
When my woman left home for a brown-eyed man
But I still don't seem to care


Good times, bad times, you know I've had my share
When my woman left home for a brown-eyed man
But I still don't seem to care

I know what it means to be alone


# Save your lyrics
Now that you're familiar with the interface, make a folder named corpora and save some songs to it! You can choose to put one song in a .txt or put in a bunch in one for each artist. We recommend the latter; why so?

In [76]:
def save_songs(filename, lyrics):
    """
    filename: name of the file
    lyrics: a string containing lyrics
    """
    with open(os.path.join("corpora", filename+".txt"), "w") as f:
        f.write(lyrics)

# Saving one song per file
save_songs("Led Zeppelin", genius.search_song("Good Times Bad Times", "The Beatles").lyrics)

# Saving multiple songs per file
def save_many_songs(artist, max):
    """
    artist: The artist's name
    max: Max number of songs to find
    """
    save_songs(artist, genius.search_artist(artist, max_songs=max, sort="popularity").to_text())

save_many_songs("Led Zeppelin", 5)
save_many_songs("pinkpantheress", 5)
 # Use a bigger number to get more songs, though it'll take longer (max is 50)

corpus = "corpora/Led Zeppelin.txt"
print(f"Selected: {corpus}")



Selected: corpora/Led Zeppelin.txt


# Mix two artists together!
Select two corpora to merge into one larger file. If you use the mixed corpus, remember to update the corpus path in the section above to "corpora/mixed.txt".

In [79]:
# ── Mix two artists together! ───────────────────────────────────────────
corpus1 = "corpora/Led Zeppelin.txt"   # change to your first artist
corpus2 = "corpora/pinkpantheress.txt"    # change to your second artist
mixed_corpus = "corpora/mixed.txt"      # this is where the mix will be saved
# ────────────────────────────────────────────────────────────────────────

# combine the two corpora into one file
with open(corpus1, "r", encoding="utf-8") as f1, \
     open(corpus2, "r", encoding="utf-8") as f2, \
     open(mixed_corpus, "w", encoding="utf-8") as out:
    out.write(f1.read())
    out.write("\n")
    out.write(f2.read())

print(f"Mixed corpus saved to {mixed_corpus}!")
print("To use it, uncommon the line below")
corpus = 'corpora/mixed.txt'

Mixed corpus saved to corpora/mixed.txt!
To use it, uncommon the line below


# N-Gram code
You don’t need to change anything in this cell, but make sure to run it so the generator works. Feel free to read through the code if you’re curious about how the model works.

In [80]:
import random

from pathlib import Path


def n_gram_make_dictionary(filename: str, n: int) -> dict[tuple, list[str]]:
    """ Return a dictionary where the keys are words in <filename> and the value for a
    key is the list of words that were found to follow the key."""
    with open(filename, 'r', encoding='utf-8') as file:
        word_lst = []
        for line in file:
            word_lst.extend(line.split())
    word_dict = {}
    key = tuple([""] * n)
    for word in word_lst:
        word_dict.setdefault(key, []).append(word)
        key = key[1:] + (word,)
    return word_dict

def n_gram_mimic_text(word_dict: dict[tuple, list[str]], num_words: int, n: int) -> str:
    key = random.choice(list(word_dict.keys()))  # start from a random point
    text = []
    recent_keys = []
    max_recent = 20
    i = 0
    while i < num_words:
        if key in word_dict:
            words = word_dict[key]
            word = random.choice(words)
            key = key[1:] + (word,)
            if key in recent_keys:
                key = random.choice(list(word_dict.keys()))
                recent_keys.clear()
            else:
                recent_keys.append(key)
                if len(recent_keys) > max_recent:
                    recent_keys.pop(0)
            text.append(word)
            i += 1
        else:
            key = random.choice(list(word_dict.keys()))
    return " ".join(text)


def n_gram_generate_text(filename: str, num_words: int, n: int) -> str:
    word_dict = n_gram_make_dictionary(filename, n)
    return n_gram_mimic_text(word_dict, num_words, n)

# Generate lyrics!
Now it’s your turn to generate lyrics. Adjust the number of words and the n‑gram size, then let your chosen corpora generate new lines.

In [81]:
# ── Generate your lyrics! ───────────────────────────────────────────────
num_words = 100   # how many words to generate
n = 2             # n-gram size (try 2, 3, or 4)
# ────────────────────────────────────────────────────────────────────────

output = n_gram_generate_text(corpus, num_words, n)
print(output)

(Me) 'Cause I fly Stockholm to LA Leave my feelings on the whispering wind? Oh And as we wind on down the road Our shadows taller than our soul There walks a lady we all call the tune Then the piper will lead us to reason And a new day will dawn for those who stand long And the forests will echo with laughter Oh-oh-oh-oh-woah-ho If there's a bustle in your hedgerow, don't be alarmed now It's just a spring clean for the May queen Yes, there are two paths you can go by, but in the footsteps of dawn


# Wrap‑Up: Share Your Best Lines
You’ve now generated your own n‑gram lyrics and poems. Take a moment to look over your results, pick your favorite lines, choose a backing track from the slides and share them with the group!
